# Phase 3E.03 — Transaction Costs, Turnover & Net Performance

## 1. Research Objective

This notebook evaluates the economic impact of portfolio turnover
and transaction costs on the 12-month TSMOM strategy.

The analysis compares gross and net performance against the
Equal-Weighted Long-Only benchmark across Train and Validation.

The Test period remains untouched until Phase 3F.

In [18]:
from pathlib import Path
import sys

project_root = Path.cwd()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if not (project_root / "src").is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
import src.data
import src.metrics
import src.research
import src.signals
import src.strategy

from matplotlib.ticker import PercentFormatter
from src.data import prepare_monthly_prices
from src.strategy import build_tsmom_positions, calculate_portfolio_weights, calculate_portfolio_exposure, calculate_gross_portfolio_returns
from src.metrics import calculate_performance_metrics

## 2. Data & Portfolio Setup

Reconstruct the 12-month TSMOM portfolio and its Equal-Weighted Long-Only benchmark using the existing research infrastructure.

$$
p_{i,t}=s_{i,t-1},\qquad w_{i,t}=\frac{p_{i,t}}{N},\quad N=16
$$

$$
r_{i,t}=\frac{P_{i,t}}{P_{i,t-1}}-1
$$

$$
R_{p,t}^{gross}=\sum_{i=1}^{N}w_{i,t}r_{i,t}
$$

$$
w_{i,t}^{B}=\frac{1}{N},\qquad R_{B,t}=\frac{1}{N}\sum_{i=1}^{N}r_{i,t}
$$

**Notation and implementation.** $i$ denotes an ETF, $t$ the holding month, $N=16$, $s_{i,t-1}$ the preceding month-end signal, and $P_{i,t}$ the adjusted month-end price. $p_{i,t}$ and $w_{i,t}$ correspond to `positions` and `portfolio_weights`; $r_{i,t}$ is `asset_returns`; $R_{p,t}^{gross}$ is `gross_portfolio_returns`. The benchmark uses $w_{i,t}^{B}$ (`benchmark_weights`) and $R_{B,t}$ (`benchmark_returns`).

In [19]:
asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

start_date = "2008-01-01"
end_date = "2022-01-01"
lookback = 12

monthly_prices = prepare_monthly_prices(tickers, start_date, end_date)

signals, positions = build_tsmom_positions(monthly_prices, lookback=lookback)

portfolio_weights = calculate_portfolio_weights(positions)

gross_exposure, net_exposure = calculate_portfolio_exposure(portfolio_weights)

print(monthly_prices.head())
print(monthly_prices.shape)

print(positions.head())
print(positions.shape)

print(portfolio_weights.dropna().head())
print(portfolio_weights.shape)

Ticker            DBA        DBB        DBC        EEM        EWJ        FXA  \
Date                                                                           
2008-01-31  31.200174  19.611866  27.033497  30.937830  36.718540  65.858284   
2008-02-29  35.054863  21.755014  30.046467  31.554821  36.170074  68.784737   
2008-03-31  30.744694  20.388248  29.771805  30.370560  35.708206  67.720993   
2008-04-30  30.871214  20.388248  31.569595  33.145889  38.335087  70.257141   
2008-05-31  29.816870  19.385418  33.750252  34.194553  39.085629  71.677910   

Ticker             FXB         FXE         FXY        HYG        IEF  \
Date                                                                   
2008-01-31  173.245621  136.197571   93.910004  30.854900  56.444172   
2008-02-29  173.586090  139.361984   96.129997  30.278101  57.143635   
2008-03-31  174.066635  145.244644  100.239998  30.564543  57.907845   
2008-04-30  175.017502  144.052856   95.949997  31.602823  56.507938   
2008-05

In [20]:
asset_returns, weighted_returns, gross_portfolio_returns = calculate_gross_portfolio_returns(monthly_prices, portfolio_weights)

print(asset_returns.head())
print(asset_returns.shape)

print(weighted_returns.head())
print(weighted_returns.shape)

print(gross_portfolio_returns.dropna().head())
print(gross_portfolio_returns.shape)

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-02-29  0.123547  0.109278  0.111453  0.019943 -0.014937  0.044436   
2008-03-31 -0.122955 -0.062825 -0.009141 -0.037530 -0.012769 -0.015465   
2008-04-30  0.004115  0.000000  0.060386  0.091382  0.073565  0.037450   
2008-05-31 -0.034153 -0.049187  0.069075  0.031638  0.019578  0.020222   

Ticker           FXB       FXE       FXY       HYG       IEF       LQD  \
Date                                                                     
2008-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-02-29  0.001965  0.023234  0.023640 -0.018694  0.012392 -0.007063   
2008-03-31  0.002768  0.042211  0.042755  0.009460  0.013373 -0.005972   
2008-04-30  0.005463 -0.008205 -0.042797  0.033970 -0.024175  0.012447   
2008-05-31  0.000468 -0.000179 -0.013

In [27]:
benchmark_weights = pd.DataFrame(1/len(portfolio_weights.columns), index=portfolio_weights.index, columns=portfolio_weights.columns)

print(benchmark_weights.head())

benchmark_returns = (asset_returns * benchmark_weights).sum(axis=1, min_count=16)

print(benchmark_returns.head())
print(benchmark_returns.shape)

Ticker         DBA     DBB     DBC     EEM     EWJ     FXA     FXB     FXE  \
Date                                                                         
2008-01-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   
2008-02-29  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   
2008-03-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   
2008-04-30  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   
2008-05-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   

Ticker         FXY     HYG     IEF     LQD     SPY     TLT     USO     VGK  
Date                                                                        
2008-01-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  
2008-02-29  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  
2008-03-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  
2008-04-30  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  

## 3. Portfolio Turnover

### 3.1 Signal-Driven Turnover

Signal-driven turnover measures changes in target portfolio weights, excluding weight drift caused by asset returns.

$$
TO_t^{signal}=\sum_{i=1}^{N}|w_{i,t}-w_{i,t-1}|
$$

For $n$ valid monthly observations, annualized average turnover is $12\left(\sum_t TO_t^{signal}/n\right)$. The convention measures total traded notional without dividing by two.

**Notation and implementation.** $w_{i,t}$ and $w_{i,t-1}$ are consecutive signed target weights. $TO_t^{signal}$ is the total absolute target-weight change for month $t$, stored in `signal_turnover`. `weight_changes` stores signed differences before the absolute value. The annualized mean is computed using `signal_turnover.loc["2010":"2021"].mean() * 12`.

In [36]:
weight_changes = portfolio_weights.diff()

signal_turnover = weight_changes.abs().sum(axis=1, min_count=16)

print(signal_turnover.loc["2010":"2021"].mean())
print(signal_turnover.loc["2010":"2021"].max())
print(signal_turnover.loc["2010":"2021"].median())
print(signal_turnover.loc["2010":"2021"].mean() * 12)

0.21961805555555555
1.0
0.1875
2.6354166666666665


**Results (2010–2021):** average monthly signal-driven turnover 21.96%, median 18.75%, maximum 100.00%, annualized average 263.54%. These estimates exclude weight drift.

### 3.2 Weight Drift

Asset returns cause portfolio weights to deviate from their initial target allocation before rebalancing. Under the gross-return model, assuming no additional trades during the holding month:

$$
w_{i,t}^{drift}=w_{i,t}\frac{1+r_{i,t}}{1+R_{p,t}^{gross}}
$$

The expression applies to signed long and short weights, provided portfolio wealth remains positive. It excludes financing, borrow fees and transaction costs.

**Notation and implementation.** $w_{i,t}$ is the beginning-of-month signed target weight (`portfolio_weights`), $r_{i,t}$ is the ETF return (`asset_returns`), and $R_{p,t}^{gross}$ is the total gross portfolio return (`gross_portfolio_returns`). The numerator marks the position to market; the denominator scales it by current portfolio wealth. $w_{i,t}^{drift}$ is the end-of-month pre-rebalancing weight stored in `drifted_weights`.

In [37]:
drifted_weights = (portfolio_weights * (1 + asset_returns)).div(
    1 + gross_portfolio_returns, axis=0
)

print(drifted_weights.dropna().head())
print(drifted_weights.shape)

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2009-02-28 -0.057034 -0.062531 -0.056831 -0.056446 -0.052649 -0.060678   
2009-03-31 -0.068198 -0.074534 -0.068839 -0.076867 -0.070506 -0.071663   
2009-04-30 -0.068243 -0.069642 -0.066018 -0.076366 -0.071187 -0.069207   
2009-05-31 -0.075425 -0.071199 -0.079519 -0.079296 -0.075306 -0.075460   
2009-06-30 -0.056914 -0.064957 -0.060733 -0.060967 -0.063110 -0.062875   

Ticker           FXB       FXE       FXY       HYG       IEF       LQD  \
Date                                                                     
2009-02-28 -0.059519 -0.059610  0.055399 -0.054370  0.059778 -0.057087   
2009-03-31 -0.065952 -0.068956  0.064808 -0.067030  0.067955 -0.066086   
2009-04-30 -0.068064 -0.065838  0.066328 -0.075234  0.064270 -0.067919   
2009-05-31 -0.074736 -0.073083  0.070767 -0.070347  0.066973 -0.069978   
2009-06-30 -0.063471 -0.061912  0.061

### 3.3 Actual Portfolio Turnover — Pending Implementation

Actual turnover compares the current target weights with the previous month's post-return weights:

$$
TO_t^{actual}=\sum_{i=1}^{N}|w_{i,t}-w_{i,t-1}^{drift}|
$$

This is the planned gross-model turnover convention, not a completed calculation. Initial entry handling, benchmark turnover, transaction costs and net performance remain to be implemented. Test remains excluded.

**Planned output (not yet coded).** $w_{i,t}$ is the new target allocation for holding month $t$, and $w_{i,t-1}^{drift}$ is the prior month-end pre-trade allocation from `drifted_weights.shift(1)`. $TO_t^{actual}$ will measure total absolute traded notional as a fraction of portfolio wealth. No actual-turnover output variable or result exists yet.